# Step 3: 常用可视化与 ISP API

一些接口之前已经用过，但这里详细展开
这一节直接调用 TianmouCV 的 ISP / 可视化接口：

- `default_rgb_isp`
- `gray_world_awb`
- `ACESToneMapping`
- `vizDiff`
- `SD2XY`

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / 'tianmoucv').exists() and (candidate / 'tianmoucv_example').exists():
        repo_root = candidate
        break
if repo_root is None:
    raise RuntimeError('找不到 tianmoucv 仓库根目录，请从仓库内打开本 Notebook。')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import matplotlib.pyplot as plt
import torch


In [ ]:
from tianmoucv.data import TianmoucDataReader, cone_tmdat_to_npy
from tianmoucv.isp import ACESToneMapping, SD2XY, default_rgb_isp, gray_world_awb, vizDiff
from tianmoucv.utils import ensure_tutorial_dataset

info = ensure_tutorial_dataset()
dataset = TianmoucDataReader(str(info.sample_dir), N=1, print_info=False)
sample = dataset[0]

## 1. RGB ISP 与白平衡

In [ ]:
cone_file = next((info.sample_dir / 'cone').glob('*.tmdat'))
cone_raw, _ = cone_tmdat_to_npy(str(cone_file), idx=0, mode='RAW', print_info=False)
cone_rgb, _ = default_rgb_isp(cone_raw, blc=0)
cone_awb = gray_world_awb(cone_rgb)
cone_tone = ACESToneMapping(np.clip(sample['F0_HDR'].numpy(), 0, None))

plt.figure(figsize=(15, 4))
plt.subplot(1, 3, 1)
plt.imshow(cone_rgb)
plt.title('default_rgb_isp')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(cone_awb)
plt.title('gray_world_awb')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(np.clip(cone_tone, 0, 1))
plt.title('ACESToneMapping(F0_HDR)')
plt.axis('off')
plt.tight_layout()
plt.show()

## 2. 用 `vizDiff` 显示 TD/SD 差分信号

In [ ]:
t_index = 12
raw_frame = sample['rawDiff'][:, t_index]

def render_diff(x, gain=8):
    image = vizDiff(torch.as_tensor(x), gain=gain, bg_color='black')/255.0
    if torch.is_tensor(image):
        image = image.detach().cpu().numpy()
    return image

plt.figure(figsize=(12, 4))
for i, title in enumerate(['TD', 'SDL', 'SDR']):
    plt.subplot(1, 3, i + 1)
    plt.imshow(render_diff(raw_frame[i]))
    plt.title(title)
    plt.axis('off')
plt.tight_layout()
plt.show()

## 3. 从 `SDL/SDR` 变换到 `Ix/Iy`

`SD2XY` 是从原始差分排列走向经典图像处理与下游算法的关键桥梁。

In [ ]:
sd = sample['rawDiff'][1:, t_index].permute(1, 2, 0) / 128.0
ix, iy = SD2XY(sd)

plt.figure(figsize=(15, 4))
plt.subplot(1, 3, 1)
plt.imshow(render_diff(sample['rawDiff'][1, t_index]))
plt.title('SDL')
plt.axis('off')

plt.subplot(1, 3, 2)
ix_blue_green = vizDiff(ix, gain=4, bg_color='black')/255.0
plt.imshow(ix_blue_green.numpy(), cmap='gray')
plt.title('Ix')
plt.axis('off')

plt.subplot(1, 3, 3)
iy_blue_green = vizDiff(iy, gain=4, bg_color='black')/255.0
plt.imshow(iy_blue_green.numpy(), cmap='gray')
plt.title('Iy')
plt.axis('off')
plt.tight_layout()
plt.show()

print('Ix shape =', tuple(ix.shape))
print('Iy shape =', tuple(iy.shape))